In [1]:
# ==================================================================================
# EEG-TO-TEXT MODEL TRAINING SCRIPT (BLIP CAPTION VERSION)
# Uses: EEG data + BLIP captions (tokenized) as input
# Output: Predicted text (trained on BLIP captions)
# Removed: Metadata prediction, text_caption column, metadata column
# ==================================================================================

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import evaluate as hf_evaluate


/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# ==================================================================================
# CONSTANTS
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_blip_only_final.h5"  # NEW DATASET PATH
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:

# ==================================================================================
# GRANGER CAUSALITY
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

In [4]:
# ==================================================================================
# DATASET AND DATALOADER
# ==================================================================================
class EEGBLIPDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]
            print(f"Dataset loaded: {self.n_samples} samples")
            print(f"Available columns: {list(f.keys())}")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        # Load only EEG and BLIP captions (already tokenized)
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        blip_caption = torch.from_numpy(self.h5_file['blip_input_ids'][idx].astype(np.int64))

        return eeg, blip_caption

def collate_batch(batch):
    eeg_list, blip_list = [], []
    for eeg, blip in batch:
        eeg_list.append(eeg)
        blip_list.append(blip)

    eeg_batch = torch.stack(eeg_list, dim=0)
    blip_padded = pad_sequence(blip_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), blip_padded

In [5]:

# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)
        print(f"Encoder RNN input size: {enc_hidden}")

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden

class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

class CrossAttention(nn.Module):
    """Cross-attention between EEG and BLIP features"""
    def __init__(self, eeg_dim, blip_dim, hidden_dim=256, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim
        self.head_dim = hidden_dim // num_heads
        
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        
        self.query = nn.Linear(eeg_dim, hidden_dim)
        self.key = nn.Linear(blip_dim, hidden_dim)
        self.value = nn.Linear(blip_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.1)
        
        print(f"CrossAttention: {num_heads} heads, hidden_dim={hidden_dim}")
    
    def forward(self, eeg_features, blip_features):
        """
        Args:
            eeg_features: [batch, eeg_dim]
            blip_features: [batch, blip_dim]
        Returns:
            attended: [batch, hidden_dim] - BLIP features attended by EEG
        """
        batch_size = eeg_features.size(0)
        
        # Linear projections and reshape for multi-head attention
        Q = self.query(eeg_features).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(blip_features).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(blip_features).view(batch_size, 1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attended = torch.matmul(attn_weights, V)
        attended = attended.transpose(1, 2).contiguous().view(batch_size, self.hidden_dim)
        attended = self.out(attended)
        
        return attended

class BLIPEncoder(nn.Module):
    """Encodes BLIP captions (already tokenized) into fixed-size features"""
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=256, num_layers=2, dropout=0.2, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn = nn.GRU(emb_dim, hidden_dim, num_layers, 
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim
        print(f"BLIPEncoder output dimension: {self.output_dim}")

    def forward(self, blip_tokens):
        """
        Args:
            blip_tokens: [batch, seq_len] - tokenized BLIP captions
        Returns:
            blip_features: [batch, hidden_dim] - encoded BLIP features
        """
        embedded = self.dropout(self.embedding(blip_tokens))
        _, hidden = self.rnn(embedded)
        # Take the last layer's hidden state
        blip_features = hidden[-1]
        return blip_features

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, blip_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        # RNN input: token embedding + attention context + BLIP features + global EEG context
        self.rnn_input_dim = emb_dim + enc_dim + blip_features_dim + enc_dim
        print(f"Decoder RNN input dimension: {self.rnn_input_dim}")
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, blip_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        blip_features_unsqueezed = blip_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        rnn_input = torch.cat((
            embedded,
            context_permuted,
            blip_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.blip_encoder = BLIPEncoder(text_vocab_size, emb_dim, dec_hidden, dec_layers, dropout, pad_id)
        
        blip_features_dim = self.blip_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                                 blip_features_dim, dec_layers, pad_id, dropout)

    def forward(self, eeg, blip_caption, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        """
        Args:
            eeg: [batch, channels, time] - EEG data
            blip_caption: [batch, seq_len] - tokenized BLIP captions
            target_text: [batch, seq_len] - target text (same as blip_caption for training)
            edge_index, edge_attr: Granger causality graph
            teacher_forcing_ratio: probability of using ground truth tokens
        Returns:
            text_logits: [batch, target_len-1, vocab_size]
        """
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        # Encode EEG
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        # Get global EEG context
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        # Encode BLIP caption
        blip_features = self.blip_encoder(blip_caption)

        # Text generation loop
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                blip_features,
                global_eeg_context
            )

            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2)



In [6]:
# ==================================================================================
# TRAINING AND EVALUATION
# ==================================================================================
def train_one_epoch(model, loader, optimizer, text_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, blip_b in progress_bar:
        eeg_b, blip_b = eeg_b.to(device), blip_b.to(device)

        optimizer.zero_grad()

        # Forward pass: use blip_caption as both input and target
        text_logits = model(eeg_b, blip_b, blip_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.5)

        # Calculate loss
        loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), blip_b[:, 1:].reshape(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, text_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, blip_b in progress_bar:
        eeg_b, blip_b = eeg_b.to(device), blip_b.to(device)

        # Forward pass with no teacher forcing
        text_logits = model(eeg_b, blip_b, blip_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0)

        loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), blip_b[:, 1:].reshape(-1))
        
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    return total_loss / len(loader)



In [7]:
# ==================================================================================
# MAIN EXECUTION
# ==================================================================================

if __name__ == "__main__":
    # Create dataset and loaders
    dataset = EEGBLIPDataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val = int(N * VAL_PCT)
    n_test = N - n_train - n_val
    g = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

    # Create Granger matrix
    print("Creating Granger Causality matrix...")
    try:
        eeg_b, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]

        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index,
            edge_attr=granger_edge_attr,
            num_nodes=num_channels,
            fill_value=1.0
        )

        if granger_edge_attr is None:
            granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)

        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
        print(f"Granger matrix created: {granger_edge_index.shape}")

    except Exception as e:
        print(f"Error creating Granger matrix: {e}. Using fallback.")
        num_channels = 62
        edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
        granger_edge_index = edge_index.to(torch.long).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

    # Instantiate model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        pad_id=PAD_ID,
        dropout=0.2,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2
    ).to(device)

    print(f"Model instantiated on '{device}'.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Setup training
    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

    # Training loop
    EPOCHS = 40
    best_val_loss = float('inf')

    print("\n--- Starting Training ---")

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer, text_criterion, granger_edge_index, granger_edge_attr)
        val_loss = evaluate(model, val_loader, text_criterion, granger_edge_index, granger_edge_attr)

        scheduler.step(val_loss)
        end_time = time.time()
        epoch_mins = int((end_time - start_time) / 60)
        epoch_secs = int((end_time - start_time) % 60)

        print(f'\nEpoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'\tTrain Loss: {train_loss:.4f}')
        print(f'\t  Val Loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model_save_path = 'eeg-blip-model-2.pt'
            torch.save(model.state_dict(), model_save_path)
            print(f"\t-> Val loss decreased. Saving best model to '{model_save_path}'")
        else:
            print("\t-> Val loss did not improve.")

    print("\n--- Training Complete ---")

    # ==================================================================================
    # INFERENCE
    # ==================================================================================
    
    checkpoint_path = 'eeg-blip-model-2.pt'
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\nBest model '{checkpoint_path}' loaded for inference.")

    @torch.no_grad()
    def generate_text(model, eeg_signal, blip_caption, edge_index, edge_attr,
                      k=5, penalty_alpha=0.3, context_beta=0.7, max_len=100):
        """
        Generate text from EEG and BLIP caption
        
        Args:
            eeg_signal: [channels, time] - single EEG sample
            blip_caption: [seq_len] - tokenized BLIP caption
            edge_index, edge_attr: Granger graph
        Returns:
            predicted_text: str
        """
        model.eval()
        eeg_signal = eeg_signal.unsqueeze(0).to(device)
        blip_caption = blip_caption.unsqueeze(0).to(device)

        # Encode EEG
        encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
        
        # Get global EEG context
        hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        # Encode BLIP caption
        blip_features = model.blip_encoder(blip_caption)

        # Initialize decoder
        decoder_hidden = model.decoder.init_hidden(encoder_hidden)

        # Generation loop
        generated_ids = torch.tensor([SOS_ID], device=device)
        for step in range(max_len):
            input_token = generated_ids[-1].unsqueeze(0)

            prediction, new_hidden, attention_context = model.decoder(
                input_token,
                decoder_hidden,
                encoder_outputs,
                blip_features,
                global_eeg_context
            )
            decoder_hidden = new_hidden
            
            model_log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
            topk_model_log_probs, topk_ids = torch.topk(model_log_probs, k)
            
            current_seq_len = generated_ids.shape[0]
            prev_token_embeddings = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
            candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
            
            sim_matrix = torch.matmul(candidate_token_embeddings, prev_token_embeddings.t())
            degeneration_penalty = torch.zeros(k, device=device)
            if current_seq_len > 1:
                degeneration_penalty, _ = torch.max(sim_matrix, dim=-1)
                
            current_decoder_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
            context_agreement_score = torch.matmul(candidate_token_embeddings, current_decoder_state)
            
            final_score = topk_model_log_probs + context_beta * context_agreement_score - penalty_alpha * degeneration_penalty
            
            best_next_token_idx = torch.argmax(final_score)
            next_token_id = topk_ids[best_next_token_idx]

            generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)])
            if next_token_id.item() == EOS_ID:
                break
                
        if generated_ids.numel() > 1:
            predicted_text_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
            predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
        else:
            predicted_text = ""

        return predicted_text

    

Dataset loaded: 28000 samples
Available columns: ['blip_input_ids', 'eeg']
Creating Granger Causality matrix...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger matrix created: torch.Size([2, 2822])
Encoder RNN input size: 256
BLIPEncoder output dimension: 256
Decoder RNN input dimension: 1536
Model instantiated on 'cuda'.
Total parameters: 28,350,266

--- Starting Training ---


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 06 | Time: 7m 52s
	Train Loss: 3.2081
	  Val Loss: 3.8282
	-> Val loss decreased. Saving best model to 'eeg-blip-model-2.pt'


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
# =========================
# CELL 2: INFERENCE ONLY
# =========================

# 1. Rebuild dataset (same as training)
dataset = EEGBLIPDataset(H5_FILE_PATH)

N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val = int(N * VAL_PCT)
n_test = N - n_train - n_val

g = torch.Generator().manual_seed(42)
_, _, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

# 2. Load Granger Matrix (IMPORTANT: RECOMPUTE or LOAD FROM FILE)
eeg_b, _ = next(iter(DataLoader(test_ds, batch_size=1, collate_fn=collate_batch)))
granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
granger_edge_index, granger_edge_attr = add_self_loops(
    granger_edge_index,
    edge_attr=granger_edge_attr,
    num_nodes=eeg_b.shape[1],
    fill_value=1.0
)
granger_edge_index = granger_edge_index.to(torch.long).to(device)
granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)

# 3. Load model
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    pad_id=PAD_ID,
    dropout=0.2,
    enc_hidden=256,
    dec_hidden=256,
    emb_dim=256,
    dec_layers=2
).to(device)

model.load_state_dict(torch.load("eeg-blip-model-2.pt", map_location=device))
model.eval()

print("Model loaded successfully.")

# 4. Inference function
@torch.no_grad()
def generate_text(model, eeg_signal, blip_caption, edge_index, edge_attr,
                  k=5, penalty_alpha=0.3, context_beta=0.7, max_len=100):

    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    blip_caption = blip_caption.unsqueeze(0).to(device)

    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)

    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

    blip_features = model.blip_encoder(blip_caption)
    decoder_hidden = model.decoder.init_hidden(encoder_hidden)

    generated_ids = torch.tensor([SOS_ID], device=device)

    for _ in range(max_len):
        token = generated_ids[-1].unsqueeze(0)

        prediction, new_hidden, attn = model.decoder(
            token, decoder_hidden,
            encoder_outputs, blip_features, global_eeg_context
        )
        decoder_hidden = new_hidden

        # Top-k + penalties
        log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
        topk_log_probs, topk_ids = torch.topk(log_probs, k)

        prev_emb = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
        cand_emb = F.normalize(model.decoder.embedding(topk_ids), dim=-1)

        sim = (cand_emb @ prev_emb.t())
        deg_penalty, _ = torch.max(sim, dim=-1)

        dec_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
        ctx_score = cand_emb @ dec_state

        score = topk_log_probs + context_beta * ctx_score - penalty_alpha * deg_penalty
        next_token = topk_ids[torch.argmax(score)]

        generated_ids = torch.cat([generated_ids, next_token.unsqueeze(0)])

        if next_token.item() == EOS_ID:
            break

    decoded_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
    return tokenizer.decode(decoded_ids.tolist(), skip_special_tokens=True)

# 5. Run inference
for i in range(10):
    eeg_sample, blip_caption = test_ds[i]

    pred = generate_text(model, eeg_sample, blip_caption,
                         granger_edge_index, granger_edge_attr)

    true_text = tokenizer.decode(blip_caption.tolist(), skip_special_tokens=True)

    print(f"\nSAMPLE {i+1}")
    print("TRUE :", true_text)
    print("PRED :", pred)


Dataset loaded: 28000 samples
Available columns: ['blip_input_ids', 'eeg']
Encoder RNN input size: 256
BLIPEncoder output dimension: 256
Decoder RNN input dimension: 1536
Model loaded successfully.

SAMPLE 1
TRUE : a group of yellow fish swimming in the ocean
PRED : a person is a a in the water

SAMPLE 2
TRUE : a river with a waterfall in the middle
PRED : a person is a a in the water

SAMPLE 3
TRUE : fireworks in the dark sky
PRED : a person is a a in the water

SAMPLE 4
TRUE : a panda bear is laying down on the ground
PRED : a person is a a in the water

SAMPLE 5
TRUE : a little rabbit is in the tub
PRED : a person is a a in the water

SAMPLE 6
TRUE : a large elephant walking through a field of tall grass
PRED : a person is a a in the water

SAMPLE 7
TRUE : a mountain scene with a river and mountains
PRED : a person is a a in the water

SAMPLE 8
TRUE : a city with a bridge and buildings
PRED : a person is a a in the water

SAMPLE 9
TRUE : a purple flower with green leaves in the back